In [30]:
import os
from dotenv import load_dotenv

load_dotenv()

print("Key loaded:", os.environ.get("OPENAI_API_KEY", "NOT FOUND")[:10], "...")

Key loaded: sk-proj-k5 ...


## STATE

In [31]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    ticket_category: str

## TOOLS

In [32]:
from langchain_core.tools import tool

@tool
def classify_ticket(ticket_text: str) -> dict:
    '''
    Classifies a customer support ticket into a category.
    Use this when you receive a support ticket.

    Args:
        ticket_text: The raw text of the customer support ticket
    '''
    text_lower = ticket_text.lower()

    billing_keywords   = ["invoice", "charge", "payment", "refund", "billing", "subscription"]
    technical_keywords = ["error", "crash", "bug", "not working", "login", "password", "404"]

    billling_score = sum(1 for kw in billing_keywords if kw in text_lower)
    technical_score = sum(1 for kw in technical_keywords if kw in text_lower)

    if billling_score > technical_score and billling_score > 0:
        category = 'billing'
    elif technical_score > 0:
        category = 'technical'
    else:
        category = 'general'

    return {'category': category}

# No LLM involved Quick test
result = classify_ticket.invoke({"ticket_text": "I was charged twice on my invoice"})
print(result)

{'category': 'billing'}


## BINDING

In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o', temperature=0) # temperature=0 for factual response

# binding the LLM to the tools
tools = [classify_ticket] # list of tools
llm_with_tools = llm.bind_tools(tools) # binding to the LLM

# quick test
from langchain_core.messages import HumanMessage
test_response = llm_with_tools.invoke([HumanMessage(content='I was charged twice on my invoice')])

print('Type:', type(test_response))
print('Tool calls: ', test_response.tool_calls)
print('Content: ', test_response.content)

Type: <class 'langchain_core.messages.ai.AIMessage'>
Tool calls:  [{'name': 'classify_ticket', 'args': {'ticket_text': 'I was charged twice on my invoice'}, 'id': 'call_kbtEbzw94LZO3SbZVscwUiZ2', 'type': 'tool_call'}]
Content:  


## AGENT NODE

In [34]:
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = """You are a customer support triage agent.
When you receive a support ticket, use the classify_ticket tool to categorize it.
After classification, summarize what the customer needs in one sentence."""

def agent_node(state: AgentState) -> dict:
    print('\n[Agent Node] LLM is thinking...')

    # pull conversation history from the state
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state['messages']

    # call the LLM
    response = llm_with_tools.invoke(messages)

    print('[Agent Node] Tool calls: ', response.tool_calls)
    print('[Agent Node] Content: ', response.content)

    # write LLM response back to state
    # add_messages will update, not overwrite to the state
    return {'messages': [response]}

## TOOL NODE

In [35]:
import json
from langchain_core.messages import ToolMessage

# tool lookup dict - maps tool name to the actual function
tool_map = {t.name: t for t in tools}

def tool_node(state: AgentState) -> dict:
    print('\n[Tool Node] Running tool...')

    # the last message is the LLM's response with tool_calls
    last_message = state['messages'][-1]

    tool_messages = []
    category = ''

    for tool_call in last_message.tool_calls:
        # look up by the tool name in tool_map and run it
        tool_fn = tool_map[tool_call['name']]
        result = tool_fn.invoke(tool_call['args'])

        print(f"[Tool Node] {tool_call['name']} returned: {result}")

        # capture the category if it was in classify_ticket
        if tool_call['name'] == 'classify_ticket':
            category = result['category']

        # wrap result in ToolMessage so LLM can read it
        tool_messages.append(
            ToolMessage(
                content=json.dumps(result),
                tool_call_id=tool_call['id']
            )
        )

        return {
            'messages': tool_messages,
            'ticket_category': category
        }

## THE ROUTER

In [36]:
from langgraph.graph import END

def router(state: AgentState) -> str:
    last_message = state['messages'][-1]

    if last_message.tool_calls:
        print('\n[Router] Tool call detected -> going to tool_node')
        return 'tool_node'
    
    print('\n[ROuter] No Tool call -> Going to END')
    return END

## FLOW

```markdown
START
  ↓
agent_node  →  router  →  "tool_node"  →  tool_node
                       ↘                      ↓
                        END            back to agent_node
```

## ASSEMBLING THE GRAPH

In [37]:
from langgraph.graph import StateGraph

# 1. Create the graph with our state schema
graph_builder = StateGraph(AgentState)

# 2. Register Nodes
graph_builder.add_node('agent_node', agent_node)
graph_builder.add_node('tool_node', tool_node)

# 3. Entry point - always starts here
graph_builder.set_entry_point('agent_node')

# 4. Conditional edge - after agent_node, call router to decide the next node
graph_builder.add_conditional_edges("agent_node", router, {"tool_node": "tool_node", END: END})

# 5. Fixed edge - after tool_node, always go back to edge_node
graph_builder.add_edge("tool_node", "agent_node")

# 6. compile into a runnable
graph = graph_builder.compile()

print('Graph compiled successfully...')

Graph compiled successfully...


In [38]:
def run_triage(ticket_text: str):
    print("=" * 50)
    print(f"TICKET: {ticket_text}")
    print("=" * 50)

    # initial state — just the user's message, category is empty
    initial_state = {
        "messages": [HumanMessage(content=ticket_text)],
        "ticket_category": ""
    }

    # run the graph
    final_state = graph.invoke(initial_state)

    # final answer is always the last message
    final_response = final_state["messages"][-1].content
    category = final_state["ticket_category"]

    print(f"\nCATEGORY : {category.upper()}")
    print(f"RESPONSE : {final_response}")

    return final_state

# run it
state = run_triage("I was charged twice on my invoice last month")

TICKET: I was charged twice on my invoice last month

[Agent Node] LLM is thinking...
[Agent Node] Tool calls:  [{'name': 'classify_ticket', 'args': {'ticket_text': 'I was charged twice on my invoice last month'}, 'id': 'call_DJuofuBNgIyBqSb82yG4Oubz', 'type': 'tool_call'}]
[Agent Node] Content:  

[Router] Tool call detected -> going to tool_node

[Tool Node] Running tool...
[Tool Node] classify_ticket returned: {'category': 'billing'}

[Agent Node] LLM is thinking...
[Agent Node] Tool calls:  []
[Agent Node] Content:  The customer needs assistance with a billing issue, specifically regarding being charged twice on their invoice last month.

[ROuter] No Tool call -> Going to END

CATEGORY : BILLING
RESPONSE : The customer needs assistance with a billing issue, specifically regarding being charged twice on their invoice last month.


We are building a multi-agent customer support triage system using LangGraph + GPT-4o. 

Phase 1 is complete — a single agent with one tool (classify_ticket) that classifies tickets as billing/technical/general using a StateGraph with agent_node, tool_node, and a router. 

AgentState has two fields: messages (with add_messages reducer) and ticket_category. Next is Phase 2 — three specialized agents (Classifier, Responder, Escalation) that hand off to each other in the same notebook.